# Task 4 — Vanilla logit KD with strict ternary QAT

Validation-only research notebook.  This notebook invokes production modules only; it never constructs a CIFAR-10 test loader.

**1. Environment and reproducibility**  \n**2. Locked Task 1/2/3 references**  \n**3. Task 4 scope and test firewall**  \n**4. Frozen ResNet34 teacher**  \n**5. Task 2 FP32 warm-start**  \n**6. Strict all-layer ternary conversion**  \n**7. Vanilla KD objective**  \n**8. Analytical loss tests**  \n**9. Real-batch teacher-gradient test**  \n**10. One-epoch production smoke**  \n**11. T/lambda search design**  \n**12. Stage-1 grid**  \n**13. Stage-2 survivor promotion**  \n**14. Stage-3 robustness check**  \n**15. Lambda=0 integrity control**  \n**16. Temperature analysis**  \n**17. Teacher checkpoint comparison (only if already available)**  \n**18. Optimizer/LR sensitivity**  \n**19. Quantization diagnostics**  \n**20. Final three-seed protocol**  \n**21. Test lock verification**  \n**22. Reporting checklist**

In [ ]:
from pathlib import Path
import sys, torch
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Repository:', ROOT)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert not any('test' in p.name.lower() for p in (ROOT / 'experiments' / 'task4').glob('**/*') if p.is_file()), 'Unexpected test-labelled artifact'


## Objective and non-negotiable invariants

\[L=(1-\lambda)\operatorname{CE}(y,z_s)+\lambda T^2\operatorname{KL}(\operatorname{softmax}(z_t/T)\;||\;\operatorname{softmax}(z_s/T))\]

The teacher must be `eval()`, `requires_grad=False`, and evaluated under `torch.no_grad()`.  Temperature and lambda selection is validation-only.  Feature, relation, contrastive, adaptive, DKD, and test-set procedures are out of scope.

In [ ]:
from src.kd.teacher import load_frozen_teacher, assert_teacher_frozen
from src.kd.losses import vanilla_kd_loss
from src.quant.ternary import QuantConfig
from src.training.utils import get_device
device = get_device(require_cuda=True)
teacher, teacher_ckpt, teacher_meta = load_frozen_teacher(ROOT / 'resnet34_cifar10_fp32_best.pth', device)
assert_teacher_frozen(teacher)
teacher_meta


## Preflight and execution gates

Run the preflight before any screen.  The smoke command is intentionally documented but not automatically executed by opening this notebook.  It calls the standalone Task 4 trainer and records validation-only artifacts.


In [ ]:
# In a terminal from the repository root:
# /home/vu-lab03-pc17/.local/bin/uv run python scripts/run_task4_preflight.py
# /home/vu-lab03-pc17/.local/bin/uv run python scripts/train_student_ternary.py --kd-mode vanilla --smoke --run-name task4_smoke_manual --condition smoke-validation --seeds 42
# /home/vu-lab03-pc17/.local/bin/uv run python scripts/run_task4_tlambda_screen.py --stage stage1


## Search and final reporting

Stage 1 evaluates the fixed 5×5 `T × lambda` grid for six epochs.  The driver writes rankings, promoted candidates, per-run histories, loss components, first-batch CE/KD gradient norms and cosine, teacher/student agreement, entropy, confidence, margins, quantization error, deployed sparsity, and ternary coverage.  Only then can later stages and the fixed 200-epoch × 3-seed final run be launched.

The final report must compare Task 4 against the immutable Task 3 B2-default validation baseline and clearly state that no Task 4 test result was produced.